In [ ]:
import pandas as pd

df = pd.read_csv('/datasets/games.csv')
print(df.head(5))

In [ ]:
df.columns = df.columns.str.lower()
print(df.head())

In [ ]:
print(df.info())

In [ ]:
print(df['user_score'].unique())

In [ ]:
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce')
print(df['user_score'].unique())
df.info()

In [ ]:
print(df.isnull().sum())

In [ ]:
df['year_of_release'] = df['year_of_release'].fillna(
    df.groupby('name')['year_of_release'].transform('max')
)

df['year_of_release'] = df['year_of_release'].fillna(
    df.groupby('platform')['year_of_release'].transform('median')
)

df.dropna(subset=['year_of_release'], inplace=True)
df['year_of_release'] = df['year_of_release'].astype(int)
df.info()

In [ ]:
df['total_sales'] = (
    df['na_sales'] + df['eu_sales'] + df['jp_sales'] + df['other_sales']
)
print(df.head())

In [ ]:
jogos_por_ano = df['year_of_release'].value_counts().sort_index()
print(jogos_por_ano)

In [ ]:
jogos_por_ano.plot(kind='bar', figsize=(15, 5), title='Jogos lançados por ano')

In [ ]:
vendas_por_plataforma = (
    df.groupby('platform')['total_sales'].sum().sort_values(ascending=False)
)
print(vendas_por_plataforma.head(10))

In [ ]:
top_platforms = vendas_por_plataforma.head(6).index
df_top = df[df['platform'].isin(top_platforms)]

pivot_table = df_top.pivot_table(
    index='year_of_release',
    columns='platform',
    values='total_sales',
    aggfunc='sum'
)
pivot_table.plot(figsize=(12, 6), title='Ciclo de vida das Top Plataformas')

In [ ]:
lifespan = df.groupby('platform')['year_of_release'].agg(['min', 'max'])
lifespan['life_duration'] = lifespan['max'] - lifespan['min']

top_platforms_list = df.groupby('platform')['total_sales'].sum().sort_values(ascending=False).head(10).index
avg_lifespan = lifespan.loc[top_platforms_list, 'life_duration'].mean()

print(lifespan.loc[top_platforms_list, 'life_duration'])
print(avg_lifespan)

In [ ]:
df_atual = df[df['year_of_release'] >= 2013]
print(df_atual.shape)
print(df_atual.head())

In [ ]:
vendas_recentes = df_atual.groupby('platform')['total_sales'].sum().sort_values(ascending=False)
print(vendas_recentes)

In [ ]:
from scipy import stats as st

alpha = 0.05
xbox = df_atual[(df_atual['platform'] == 'XOne') & (df_atual['user_score'].notna())]['user_score']
pc = df_atual[(df_atual['platform'] == 'PC') & (df_atual['user_score'].notna())]['user_score']

results = st.ttest_ind(xbox, pc, equal_var=False)
print(results.pvalue)